In [12]:
import pandas as pd
import numpy as np
from AutoMLPreprocessor import AutoMLPreprocessor

from sklearn.model_selection import train_test_split

In [13]:
SAMPLE_SIZE = 0.01
X_train = pd.read_csv("data/train.csv")
X_test = pd.read_csv("data/test.csv")

y_train = X_train["Churn"]
X_train = X_train.drop(columns=["Churn"])


X_train, _, y_train, _ = train_test_split(
    X_train, 
    y_train, 
    train_size=SAMPLE_SIZE,
    stratify=y_train,
    random_state=42
)                                             

In [14]:
print(X_train.columns)
print(X_train.shape)

Index(['id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges'],
      dtype='object')
(5941, 20)


In [15]:
preprocessor = AutoMLPreprocessor(
    target_col="Churn", 
    add_kmeans_features=True,
    feature_selection=True,
    add_poly_features=True, 
    remove_outliers=False,
    remove_multicollinearity=True,
    multicollinearity_threshold=0.95,
    id_threshold=0.95,
    random_state=None)

In [16]:
def create_features(df):
    df = df.copy()
    
    df['Contract_Internet'] = df['Contract'].astype(str) + '_' + df['InternetService'].astype(str)
    df['tenure_MonthlyCharges'] = df['tenure'] * df['MonthlyCharges']
    df['Senior_TechSupport'] = df['SeniorCitizen'].astype(str) + '_' + df['TechSupport'].astype(str)
    
    df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 6, 12, 24, 100], 
                                 labels=['0-6m', '6-12m', '12-24m', '24m+']).astype(str)
    df['MonthlyCharges_group'] = pd.cut(df['MonthlyCharges'], bins=[0, 35, 70, 100, 200], 
                                        labels=['low', 'medium', 'high', 'very_high']).astype(str)
    
    return df

X_train = create_features(X_train)
X_test = create_features(X_test)


X_train, y_train = preprocessor.fit_transform(X_train, y_train)
X_test = preprocessor.transform(X_test)

--- KMeans: Wytrenowano 15 klastrów ---
   -> Wybieranie top 6 cech do interakcji (ExtraTrees)...
   Wybrane kolumny do interakcji: ['FREQ_Contract_Internet', 'MonthlyCharges', 'Dist_Cluster_7', 'Dist_Cluster_6', 'FREQ_Senior_TechSupport', 'TotalCharges']


/home/JanTar/miniconda3/envs/automl/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/home/JanTar/miniconda3/envs/automl/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(
/home/JanTar/miniconda3/envs/automl/lib/python3.11/site-packages/sklearn/preprocessing/_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 4 are removed. Consider decreasing the number of bins.
  warnings.warn(


--- Współliniowość: Usunięto 55 z 137 kolumn (korelacja > 0.95) ---

--- Selekcja Cech (Hybrid) ---
-> Etap 1: Wstępne usuwanie szumu (L1 + ExtraTrees)...
   -> Ocena Liniowa (L1 Logistic Regression)...
   -> Ocena Nieliniowa (ExtraTreesClassifier)...
   L1 wybrało: 37, Trees wybrało: 48
   Wspólna decyzja: Zachowano 60 z 82 cech (usunięto 22 najsłabszych).
-> Etap 2: Liczba cech (60) <= 100. Uruchamiam SFS Backward.


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
Features: 59/1[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
Features: 58/1[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
Features: 57/1[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
Features: 56/1[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
Features: 55/1[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
Features: 54/1[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
Features: 53/1[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
Features: 52/1[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
Features: 51/1[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
Features: 50/1[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
Features: 49/1[Parallel(

-> SFS zakończony. Wybrano 50 najlepszych cech.
--- Selekcja Cech Zakończona: Tryb = sfs ---

Ostateczne kolumny:['DIV_FREQ_Contract_Internet_by_TotalCharges', 'FREQ_Contract_Internet', 'FREQ_Senior_TechSupport', 'MUL_FREQ_Contract_Internet_x_TotalCharges', 'MUL_MonthlyCharges_x_TotalCharges', 'DIV_FREQ_Senior_TechSupport_by_Dist_Cluster_7', 'DIV_FREQ_Contract_Internet_by_FREQ_Senior_TechSupport', 'DIV_TotalCharges_by_Dist_Cluster_6', 'SUB_FREQ_Contract_Internet_FREQ_Senior_TechSupport', 'MUL_Dist_Cluster_7_x_FREQ_Senior_TechSupport', 'DIV_FREQ_Senior_TechSupport_by_FREQ_Contract_Internet', 'MUL_FREQ_Contract_Internet_x_MonthlyCharges', 'DIV_TotalCharges_by_Dist_Cluster_7', 'MonthlyCharges', 'MUL_FREQ_Contract_Internet_x_Dist_Cluster_7', 'MUL_FREQ_Senior_TechSupport_x_TotalCharges', 'SQR_TotalCharges', 'DIV_TotalCharges_by_FREQ_Senior_TechSupport', 'ADD_MonthlyCharges_TotalCharges', 'MUL_FREQ_Contract_Internet_x_FREQ_Senior_TechSupport', 'LOG_MonthlyCharges', 'SQR_MonthlyCharges', 'Tot

In [18]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
import pandas as pd
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'roc_auc', 'f1', 'precision', 'recall']
results_list = []

# mało, bo to tylko testowanie
N_TRIALS = 20

optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"Rozpoczynam strojenie Optuną ({N_TRIALS} prób/model) i testowanie...\n" + "-"*60)

model_configs = ["Random Forest", "Logistic Regression", "Gradient Boosting"]

for name in model_configs:
    print(f"Strojenie hiperparametrów: {name}...")
    
    def objective(trial):
        if name == "Random Forest":
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 250, step=50),
                'max_depth': trial.suggest_int('max_depth', 3, 15),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 15),
                'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced'])
            }
            model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
            
        elif name == "Logistic Regression":
            params = {
                'C': trial.suggest_float('C', 1e-4, 1e2, log=True),
                'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced']),
                'solver': 'lbfgs', # Szybki solver
                'max_iter': 2000
            }
            model = LogisticRegression(**params, random_state=42)
            
        elif name == "Gradient Boosting":
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 250, step=50),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'max_depth': trial.suggest_int('max_depth', 3, 8),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0)
            }
            model = GradientBoostingClassifier(**params, random_state=42)
            

        score = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
        return score.mean()

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=N_TRIALS)
    
    best_params = study.best_params
    
    # Dodajemy parametry stałe, które nie były strojone
    if name == "Logistic Regression":
        best_params.update({'solver': 'lbfgs', 'max_iter': 2000})
        
    print(f"  Najlepsze parametry: {best_params}")
    
    # Rekonstrukcja modelu z najlepszymi parametrami
    if name == "Random Forest": best_model = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
    elif name == "Logistic Regression": best_model = LogisticRegression(**best_params, random_state=42)
    elif name == "Gradient Boosting": best_model = GradientBoostingClassifier(**best_params, random_state=42)

    cv_results = cross_validate(best_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    
    results_list.append({
        "Model": name,
        "Accuracy": np.mean(cv_results['test_accuracy']),
        "ROC AUC": np.mean(cv_results['test_roc_auc']),
        "F1 Score": np.mean(cv_results['test_f1']),
        "Precision": np.mean(cv_results['test_precision']),
        "Recall": np.mean(cv_results['test_recall'])
    })

results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values(by="ROC AUC", ascending=False).reset_index(drop=True)

print("\n" + "-"*60)
print("Wyniki po optymalizacji hiperparametrów (posortowane po ROC AUC):")
print(results_df.to_string(index=False))

Rozpoczynam strojenie Optuną (20 prób/model) i testowanie...
------------------------------------------------------------
Strojenie hiperparametrów: Random Forest...


  Najlepsze parametry: {'n_estimators': 200, 'max_depth': 7, 'min_samples_split': 12, 'class_weight': None}
Strojenie hiperparametrów: Logistic Regression...


/home/JanTar/miniconda3/envs/automl/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/JanTar/miniconda3/envs/automl/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/

  Najlepsze parametry: {'C': 2.706549530408277, 'class_weight': None, 'solver': 'lbfgs', 'max_iter': 2000}
Strojenie hiperparametrów: Gradient Boosting...
  Najlepsze parametry: {'n_estimators': 200, 'learning_rate': 0.026157825112002774, 'max_depth': 4, 'subsample': 0.6017203201862524}

------------------------------------------------------------
Wyniki po optymalizacji hiperparametrów (posortowane po ROC AUC):
              Model  Accuracy  ROC AUC  F1 Score  Precision   Recall
Logistic Regression  0.856759 0.913739  0.661328   0.707203 0.621849
  Gradient Boosting  0.857435 0.912077  0.669412   0.701193 0.640533
      Random Forest  0.853226 0.908531  0.655833   0.694912 0.621105
